** ETAP 1: EKSTRAKCJA DANYCH ZE ZBIORU PKLOT I BUDOWA SZEREGU CZASOWEGO **

Praca magisterska: "Analiza i porównanie metod służących do predykcji zajętości miejsc parkingowych"

In [1]:
import xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime
import pandas as pd

# 1. Definicja ścieżek
BASE_DIR = Path("d:/MAGISTERKA/mgr-research")
RAW_DATA_PATH = BASE_DIR / "data" / "raw" / "PKLot" / "PKLot"
PROCESSED_DATA_PATH = BASE_DIR / "data" / "processed" / "pklot_occupancy_timeseries.csv"
PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)


def parse_pklot_xml(xml_file: Path) -> dict | None:
    """Parsuje metadane i stopę zajętości z pojedynczego pliku XML PKLot."""
    try:
        ts = datetime.strptime(xml_file.stem, "%Y-%m-%d_%H_%M_%S")
        weather = xml_file.parent.parent.name
        parking_lot = xml_file.parent.parent.parent.name

        tree = ET.parse(xml_file)
        root = tree.getroot()
        spaces = root.findall("space")
        total_spaces = len(spaces)

        if total_spaces == 0:
            return None

        occupied_spaces = sum(1 for space in spaces if space.get("occupied") == "1")
        free_spaces = total_spaces - occupied_spaces
        occupancy_rate = (occupied_spaces / total_spaces) * 100

        return {
            "timestamp": ts,
            "weather": weather,
            "parking_lot": parking_lot,
            "total_spaces": total_spaces,
            "occupied_spaces": occupied_spaces,
            "free_spaces": free_spaces,
            "occupancy_rate": occupancy_rate,
        }
    except (ET.ParseError, OSError, ValueError) as error:
        print(f"Błąd parsowania {xml_file.name}: {error}")
        return None


# 2. Główna pętla zbierająca dane
print(f"Szukanie plików XML w: {RAW_DATA_PATH} ...")
xml_files = list(RAW_DATA_PATH.glob("*/*/*/*/*.xml"))
print(f"Znaleziono łącznie: {len(xml_files)} plików XML.")
records = []
for idx, file_path in enumerate(xml_files):
    record = parse_pklot_xml(file_path)
    if record:
        records.append(record)

    # Wyświetlanie postępu co 2500 plików
    if (idx + 1) % 2500 == 0 or (idx + 1) == len(xml_files):
        print(f"Przetworzono: {idx + 1}/{len(xml_files)} klatek...")

# 3. Konwersja do ramki danych Pandas i sortowanie chronologiczne
if not records:
    raise ValueError("Nie udało się sparsować żadnego pliku XML. Sprawdź strukturę danych.")
df_all = pd.DataFrame(records)
df_all = df_all.sort_values(by=["parking_lot", "timestamp"]).reset_index(drop=True)

# 4. Zapis do pliku CSV
df_all.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"\n[SUKCES] Szereg czasowy zapisany do: {PROCESSED_DATA_PATH}")

# 5. Podgląd danych i statystyki per parking
print("\nPodsumowanie wygenerowanego zbioru:")
print(df_all.groupby("parking_lot")[["total_spaces", "occupied_spaces", "occupancy_rate"]].agg(["count", "mean", "min", "max"]))
print("\nPierwsze 5 wierszy:")
print(df_all.head())

Szukanie plików XML w: d:\MAGISTERKA\mgr-research\data\raw\PKLot\PKLot ...
Znaleziono łącznie: 12416 plików XML.
Przetworzono: 2500/12416 klatek...
Przetworzono: 5000/12416 klatek...
Przetworzono: 7500/12416 klatek...
Przetworzono: 10000/12416 klatek...
Przetworzono: 12416/12416 klatek...

[SUKCES] Szereg czasowy zapisany do: d:\MAGISTERKA\mgr-research\data\processed\pklot_occupancy_timeseries.csv

Podsumowanie wygenerowanego zbioru:
            total_spaces                  occupied_spaces                      \
                   count   mean  min  max           count       mean min  max   
parking_lot                                                                     
PUCPR               4473  100.0  100  100            4473  42.972502   0  100   
UFPR04              3791   28.0   28   28            3791  12.169612   0   28   
UFPR05              4152   40.0   40   40            4152  23.464836   0   40   

            occupancy_rate                         
                     co